In [1]:
import numpy as np
from datetime import date, timedelta

start_date = date(2026, 6, 1)
end_date = date(2026, 6, 7)


In [2]:
ssts = []
for i in range((end_date - start_date).days):
    current_date = start_date + timedelta(days=i)
    for day in (0, 12):
        day_str = str(day).zfill(2)
        sst_example = np.load(f"../data/{current_date.strftime('%Y/%m/%d')}/{day_str}/sst.npy", allow_pickle=True)
        # sst_example = np.load(f"../data/{current_date.strftime('%m/%d')}/{day_str}/sst.npy", allow_pickle=True)
        ssts.append(sst_example.astype(np.float64))

sst_full = np.stack(ssts, axis=-1)

In [3]:
sst_full.reshape((-1, sst_full.shape[0], sst_full.shape[1])).shape

(12, 1121, 2880)

In [4]:
# Mirrors the ocean-mask construction in src/weather/model.py's
# `_mask_and_stack` (used by WeatherModel.fit): a cell counts as "ocean"
# only if it is non-NaN across every training snapshot, for every modeled
# variable (here just `sst`). Unlike the production pipeline — which then
# ravels (lat, lon) into a single `ocean_cell` Tucker mode and drops land
# cells from the tensor entirely — this notebook keeps `sst_full`'s 3D
# (lat, lon, time) shape so lat and lon stay separate Tucker modes to
# explore. Land/ice cells are instead flagged via a mask passed to
# `partial_tucker(mask=...)`, which iteratively imputes them from the
# current low-rank reconstruction instead of contributing raw NaN (or an
# arbitrary sentinel like -9999) to the decomposition.
ocean_mask = ~np.isnan(sst_full).any(axis=-1)  # (1121, 2880) bool

# partial_tucker's `mask` must match `tensor`'s shape, with 1.0 for
# observed entries and 0.0 for entries to impute.
tucker_mask = np.broadcast_to(ocean_mask[:, :, None], sst_full.shape).astype(np.float64)

# The masked HOI loop overwrites entries where tucker_mask == 0 on its
# first iteration, but tensorly computes the tensor's norm once up front
# using these raw values — NaN would poison that norm, and a sentinel like
# -9999 would inflate it. 0.0 is a neutral placeholder immediately
# replaced by the reconstruction-based imputation.
sst_filled = np.where(ocean_mask[:, :, None], sst_full, 0.0)

In [5]:
import tensorly as tl
from tensorly.decomposition import partial_tucker
from tensorly.tenalg import multi_mode_dot

In [6]:
import plotly.graph_objects as go


def plot_3d_volume(
    array,
    max_points_per_axis=60,
    opacity=0.1,
    surface_count=17,
    isomin=None,
    isomax=None,
    axis_labels=("axis 0", "axis 1", "axis 2"),
):
    """Render a 3D numpy array as an interactive Plotly volume.

    Downsamples each axis to at most `max_points_per_axis` (Volume
    rendering is O(n^3); the raw grid is usually too large to render).
    NaNs are left in place rather than filtered out, since dropping them
    would break the regular-grid assumption `go.Volume` relies on.

    Scene axis ranges are set explicitly from the actual coordinate
    extents. With NaN-heavy real data (e.g. ~30% land/ice cells in `sst`),
    Plotly's autorange can fail to find a valid extent from the trace and
    silently falls back to its default `[-1, 1]` scene — which renders as
    an apparently empty cube even though the underlying data is fine.
    """
    steps = tuple(max(1, dim // max_points_per_axis) for dim in array.shape)
    sub = array[:: steps[0], :: steps[1], :: steps[2]]

    X, Y, Z = np.mgrid[0 : sub.shape[0], 0 : sub.shape[1], 0 : sub.shape[2]]

    if isomin is None:
        isomin = np.nanpercentile(sub, 5)
    if isomax is None:
        isomax = np.nanpercentile(sub, 95)

    fig = go.Figure(
        data=go.Volume(
            x=X.flatten(),
            y=Y.flatten(),
            z=Z.flatten(),
            value=sub.flatten(),
            isomin=isomin,
            isomax=isomax,
            opacity=opacity,
            surface_count=surface_count,
        )
    )
    fig.update_layout(
        scene=dict(
            xaxis=dict(title=axis_labels[0], range=[0, sub.shape[0] - 1]),
            yaxis=dict(title=axis_labels[1], range=[0, sub.shape[1] - 1]),
            zaxis=dict(title=axis_labels[2], range=[0, sub.shape[2] - 1]),
        )
    )
    fig.show()
    return fig

In [7]:
plot_3d_volume(sst_full, axis_labels=("lat idx", "lon idx", "time idx"))

In [8]:
# Compare against sst_filled — the ocean-mask output from the mask cell
# above, with land/ice NaNs zeroed out. This is what actually gets passed
# to partial_tucker.
plot_3d_volume(sst_filled, axis_labels=("lat idx", "lon idx", "time idx"))

# Understand Tucker Decomposition

In [9]:
latitude_rank = 10  # Arbitrary rank in latitude
longitude_rank = 20  # Arbitrary rank in longitude
time_rank = sst_full.shape[-1]  # Full rank in time — sst_full is (lat, lon, time)

In [10]:
matrix_core, (latitude_factors, longitude_factors, time_factors), errors = partial_tucker(
    sst_filled,
    modes=[0, 1, 2],
    rank=[latitude_rank, longitude_rank, time_rank],
    init="random",
    tol=1e-5,
    n_iter_max=5000,
    random_state=42,
    mask=tucker_mask,
)

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
latitude_factors.shape, longitude_factors.shape, time_factors.shape

[(1121, 10), (2880, 20), (12, 12)]

In [ ]:
matrix_core.shape

(10, 20, 12)

In [ ]:
errors

[np.float64(4.911208744593004),
 np.float64(4.580466376567428),
 np.float64(4.454290800282283),
 np.float64(4.3812283268656165),
 np.float64(4.3318327492424755),
 np.float64(4.295338441940355),
 np.float64(4.266808269285928),
 np.float64(4.243625177572988),
 np.float64(4.224252810646013),
 np.float64(4.2077185100857),
 np.float64(4.193369973158775),
 np.float64(4.180749655282598),
 np.float64(4.16952497799734),
 np.float64(4.159447123907574),
 np.float64(4.150325472973263),
 np.float64(4.14201108120126),
 np.float64(4.134385624906907),
 np.float64(4.1273537723121985),
 np.float64(4.120837775814127),
 np.float64(4.114773547474178),
 np.float64(4.109107753129266),
 np.float64(4.103795623574373),
 np.float64(4.098799281528011),
 np.float64(4.094086446554152),
 np.float64(4.089629421412544),
 np.float64(4.0854042908795725),
 np.float64(4.081390282917953),
 np.float64(4.077569255211068),
 np.float64(4.073925279410609),
 np.float64(4.07044430218519),
 np.float64(4.067113867093623),
 np.float

## Resolution reduction, made concrete: one time-slice, before and after Tucker

The cells above show reconstruction *error*, but not what the compressed
*representation itself* looks like at a single point in time. The next two
cells make that concrete: the same time-slice, once in the original
`(lat, lon)` resolution, and once in the compressed `(latitude_rank,
longitude_rank)` coefficient basis Tucker found for it.

This second slice deliberately projects only through the two *spatial*
factor matrices (`latitude_factors`, `longitude_factors`), skipping the
time-mode factor (`time_factors`) even though `partial_tucker` above did
compress/rotate that mode too (`time_rank` was set to the full snapshot
count, not skipped — see the rank-choice cell above). Because time is kept
at full rank, `time_factors` is a square, orthonormal *change of basis* in
time, not a projection — so a single index into `matrix_core`'s last axis
doesn't necessarily correspond one-to-one with the same physical time index
unless that rotation happens to be close to identity. Projecting only
through the spatial factors sidesteps that ambiguity entirely and isolates
exactly the thing being asked about: how Tucker compresses *space* for one
snapshot.

In [ ]:
import matplotlib.pyplot as plt

time_index = 0

regular_slice = sst_filled[:, :, time_index]  # (1121, 2880) — full-resolution lat/lon map

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(regular_slice, origin="lower", aspect="auto", cmap="viridis")
fig.colorbar(im, ax=ax, label="sst (filled, land/ice = 0)")
ax.set_title(f"Regular space — time index {time_index}, shape {regular_slice.shape}")
ax.set_xlabel("lon idx")
ax.set_ylabel("lat idx")

In [ ]:
compressed_slice = latitude_factors.T @ regular_slice @ longitude_factors  # (10, 20)

fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(compressed_slice, origin="lower", aspect="auto", cmap="viridis")
fig.colorbar(im, ax=ax, label="Tucker core coefficient")
ax.set_title(
    f"Compressed space — same time index, shape {compressed_slice.shape}\n"
    f"({regular_slice.size:,} -> {compressed_slice.size:,} values, "
    f"~{regular_slice.size / compressed_slice.size:,.0f}x fewer)"
)
ax.set_xlabel("longitude-mode coefficient idx")
ax.set_ylabel("latitude-mode coefficient idx")

### The factor matrices are the eigenvectors of each mode

`latitude_factors` and `longitude_factors` are not arbitrary compression
weights — each is the matrix of leading left singular vectors of that
mode's unfolding (`unfold(sst_filled, mode=0)` for latitude, `mode=1` for
longitude), i.e. the eigenvectors of `X_(mode) @ X_(mode).T`. Column `k` of
`latitude_factors` is the `k`-th spatial eigenvector along the latitude
axis — the direct 3D-tensor analog of an EOF (empirical orthogonal
function) in climate science, or a POD mode in fluid dynamics.
`compressed_slice` above is literally this one time-slice's field
re-expressed as coefficients against these eigenvector bases: each entry of
`compressed_slice` says "how much of latitude-eigenvector `i` combined with
longitude-eigenvector `j` is present in this snapshot."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))

for mode_idx in range(3):
    axes[0].plot(latitude_factors[:, mode_idx], label=f"mode {mode_idx}")
axes[0].set_title("Leading latitude-mode eigenvectors")
axes[0].set_xlabel("lat idx")
axes[0].legend()

for mode_idx in range(3):
    axes[1].plot(longitude_factors[:, mode_idx], label=f"mode {mode_idx}")
axes[1].set_title("Leading longitude-mode eigenvectors")
axes[1].set_xlabel("lon idx")
axes[1].legend()

fig.tight_layout()

In [ ]:
sst_recon = multi_mode_dot(
    multi_mode_dot(sst_filled, [latitude_factors.T, longitude_factors.T], modes=[0, 1]),
    [latitude_factors, longitude_factors],
    modes=[0, 1],
)

In [ ]:
reconstruction_error = float(np.linalg.norm(sst_recon - sst_filled) / np.linalg.norm(sst_filled))


In [ ]:
reconstruction_error

0.3453435386464791

In [ ]:
plot_3d_volume(sst_recon, axis_labels=("lat idx", "lon idx", "time idx"))

In [ ]:
plot_3d_volume(sst_filled, axis_labels=("lat idx", "lon idx", "time idx"))